# Bước 3: Phân cụm DBSCAN
## Đề 13: Phân tích hội thoại trong DVKH trực tuyến
**Mục tiêu:** Nhóm các đoạn hội thoại theo chủ đề bằng DBSCAN

In [ ]:
import pandas as pd
import numpy as np
from scipy import sparse
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load data
customer_df = pd.read_csv('../data/processed/customer_messages.csv')
tfidf_matrix = sparse.load_npz('../data/processed/tfidf_matrix.npz')

print(f'Customer messages: {len(customer_df)}')
print(f'TF-IDF shape: {tfidf_matrix.shape}')

## 3.1 Gộp TF-IDF theo đoạn hội thoại

In [ ]:
# Mỗi đoạn hội thoại có thể có nhiều lượt khách hàng
# Gộp TF-IDF trung bình theo conv_id
conv_ids = customer_df['conv_id'].values
unique_convs = sorted(customer_df['conv_id'].unique())

conv_vectors = []
conv_labels = []
for cid in unique_convs:
    mask = conv_ids == cid
    avg_vec = tfidf_matrix[mask].mean(axis=0)
    conv_vectors.append(np.asarray(avg_vec).flatten())
    conv_labels.append(cid)

X = np.array(conv_vectors)
print(f'Ma trận đặc trưng theo đoạn: {X.shape}')

## 3.2 Chọn tham số eps (K-Distance Plot)

In [ ]:
# K-distance plot để chọn eps
k = 5
nn = NearestNeighbors(n_neighbors=k, metric='cosine')
nn.fit(X)
distances, _ = nn.kneighbors(X)
k_distances = np.sort(distances[:, k-1])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(k_distances, color='steelblue', linewidth=1.5)
ax.set_xlabel('Điểm dữ liệu (sắp xếp)')
ax.set_ylabel(f'{k}-Distance')
ax.set_title(f'K-Distance Plot (k={k}) — Chọn eps tại điểm "khuỷu tay"')
ax.grid(True, alpha=0.3)

# Gợi ý eps
suggested_eps = np.percentile(k_distances, 90)
ax.axhline(y=suggested_eps, color='red', linestyle='--', label=f'Gợi ý eps={suggested_eps:.3f}')
ax.legend()
plt.tight_layout()
plt.savefig('../results/figures/02_clustering/eps_selection.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Gợi ý eps (percentile 90): {suggested_eps:.4f}')

## 3.3 Thử nghiệm nhiều bộ tham số

In [ ]:
# Thử nhiều eps và min_samples
results = []
eps_range = np.arange(0.3, 1.0, 0.1)
ms_range = [3, 5, 7, 10]

for eps in eps_range:
    for ms in ms_range:
        db = DBSCAN(eps=eps, min_samples=ms, metric='cosine')
        labels = db.fit_predict(X)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = (labels == -1).sum()
        noise_pct = n_noise / len(labels) * 100
        
        sil = -1
        if n_clusters >= 2 and n_noise < len(labels) - 1:
            non_noise_mask = labels != -1
            if non_noise_mask.sum() > n_clusters:
                sil = silhouette_score(X[non_noise_mask], labels[non_noise_mask], metric='cosine')
        
        results.append({
            'eps': round(eps, 2), 'min_samples': ms,
            'n_clusters': n_clusters, 'noise': n_noise,
            'noise_pct': round(noise_pct, 1), 'silhouette': round(sil, 4)
        })

results_df = pd.DataFrame(results)
print('KẾT QUẢ THỬ NGHIỆM:')
print(results_df.to_string(index=False))

# Chọn bộ tham số tốt nhất (silhouette cao nhất, clusters >= 3)
valid = results_df[(results_df['n_clusters'] >= 3) & (results_df['silhouette'] > 0)]
if len(valid) > 0:
    best = valid.loc[valid['silhouette'].idxmax()]
    print(f'\n✅ Tham số tốt nhất: eps={best["eps"]}, min_samples={int(best["min_samples"])}')
    print(f'   Clusters={int(best["n_clusters"])}, Noise={best["noise_pct"]}%, Silhouette={best["silhouette"]}')
    BEST_EPS = best['eps']
    BEST_MS = int(best['min_samples'])
else:
    print('\n⚠️ Không tìm được bộ tham số tối ưu, dùng mặc định')
    BEST_EPS = 0.7
    BEST_MS = 5

## 3.4 Chạy DBSCAN với tham số tối ưu

In [ ]:
dbscan = DBSCAN(eps=BEST_EPS, min_samples=BEST_MS, metric='cosine')
cluster_labels = dbscan.fit_predict(X)

n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise = (cluster_labels == -1).sum()

print(f'Số cụm tìm được: {n_clusters}')
print(f'Số noise points: {n_noise} ({n_noise/len(cluster_labels)*100:.1f}%)')
print()
print('Phân phối cụm:')
for c in sorted(set(cluster_labels)):
    count = (cluster_labels == c).sum()
    label = 'NOISE' if c == -1 else f'Cụm {c}'
    print(f'  {label:10s}: {count:4d} đoạn ({count/len(cluster_labels)*100:.1f}%)')

## 3.5 Trực quan hóa t-SNE

In [ ]:
# t-SNE giảm chiều xuống 2D
tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(X)-1))
X_2d = tsne.fit_transform(X)

fig, ax = plt.subplots(figsize=(12, 8))
scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=cluster_labels,
                     cmap='tab20', s=15, alpha=0.7)

# Đánh dấu noise
noise_mask = cluster_labels == -1
ax.scatter(X_2d[noise_mask, 0], X_2d[noise_mask, 1],
           c='gray', s=10, alpha=0.3, label='Noise')

ax.set_title(f'DBSCAN Clustering (t-SNE) — {n_clusters} cụm, eps={BEST_EPS}, min_samples={BEST_MS}')
ax.legend()
plt.colorbar(scatter, label='Cluster ID')
plt.tight_layout()
plt.savefig('../results/figures/02_clustering/cluster_tsne.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.6 Phân tích nội dung từng cụm

In [ ]:
# Map cluster labels về dataframe
conv_cluster_map = dict(zip(conv_labels, cluster_labels))

# Load full data
full_df = pd.read_csv('../data/processed/cleaned_data.csv')
full_df['cluster_id'] = full_df['conv_id'].map(conv_cluster_map).fillna(-1).astype(int)

# Top keywords mỗi cụm
vocab = pd.read_csv('../data/processed/tfidf_vocab.csv')['word'].values

print('TOP KEYWORDS MỖI CỤM:')
print('=' * 60)
for c in sorted(set(cluster_labels)):
    if c == -1:
        continue
    mask = cluster_labels == c
    cluster_mean = X[mask].mean(axis=0)
    top_indices = cluster_mean.argsort()[-10:][::-1]
    top_words = [vocab[i] for i in top_indices if i < len(vocab)]
    
    # Topic gốc phổ biến nhất trong cụm
    cluster_convs = [conv_labels[i] for i in range(len(conv_labels)) if cluster_labels[i] == c]
    cluster_topics = full_df[full_df['conv_id'].isin(cluster_convs)].drop_duplicates('conv_id')['topic_name'].value_counts().head(3)
    
    print(f'\nCụm {c} ({mask.sum()} đoạn):')
    print(f'  Keywords: {", ".join(top_words)}')
    print(f'  Top chủ đề gốc:')
    for topic, cnt in cluster_topics.items():
        print(f'    - {topic}: {cnt}')

## 3.7 Đánh giá & Heatmap

In [ ]:
# Silhouette Score
non_noise = cluster_labels != -1
if non_noise.sum() > 0 and len(set(cluster_labels[non_noise])) > 1:
    sil = silhouette_score(X[non_noise], cluster_labels[non_noise], metric='cosine')
    print(f'Silhouette Score (không noise): {sil:.4f}')
else:
    print('Không đủ cụm để tính Silhouette Score')

# Heatmap: Cluster vs Topic
cluster_conv_df = full_df.drop_duplicates('conv_id')[['conv_id', 'topic_name', 'cluster_id']]
top_topics = cluster_conv_df['topic_name'].value_counts().head(12).index
subset = cluster_conv_df[cluster_conv_df['topic_name'].isin(top_topics)]
subset = subset[subset['cluster_id'] != -1]

if len(subset) > 0:
    ct = pd.crosstab(subset['topic_name'], subset['cluster_id'])
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.heatmap(ct, annot=True, fmt='d', cmap='YlOrRd', ax=ax)
    ax.set_title('Heatmap: Chủ đề gốc × Cụm DBSCAN')
    ax.set_xlabel('Cluster ID')
    ax.set_ylabel('Chủ đề gốc')
    plt.tight_layout()
    plt.savefig('../results/figures/02_clustering/cluster_topic_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()

## 3.8 Lưu kết quả

In [ ]:
full_df.to_csv('../data/final/clustered_data.csv', index=False, encoding='utf-8-sig')
print('✅ Saved: clustered_data.csv')

# Lưu summary
cluster_summary = cluster_conv_df.groupby('cluster_id').agg(
    count=('conv_id', 'count'),
    top_topic=('topic_name', lambda x: x.value_counts().index[0] if len(x) > 0 else '')
).reset_index()
cluster_summary.to_csv('../results/tables/cluster_summary.csv', index=False)
print('✅ Saved: cluster_summary.csv')
print()
print(cluster_summary)